In [2]:
print("GPU Information:")

!nvidia-smi

GPU Information:
Tue Mar 24 05:42:07 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   36C    P0             66W /  400W |    5806MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+------------------------------

In [3]:
from google.colab import drive
import os

# 1. Mount the drive
drive.mount('/content/drive')

# 2. Paths
SPARKYLLM = "/content/drive/MyDrive/sparkyllm"
TOKENIZER_OUT = os.path.join(SPARKYLLM, "datasets_pretrain", "tokenizer_out")

# 3. Import training files from tokenizer pipeline output
!cp "{TOKENIZER_OUT}/train_long.bin" "/content/train_long.bin"
!cp "{TOKENIZER_OUT}/train_long_meta.json" "/content/train_long_meta.json"
!cp "{TOKENIZER_OUT}/tokenizer.json" "/content/tokenizer.json"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
#!/usr/bin/env python3
import os

# ==================== 0. MEMORY OPTIMIZATIONS (MUST BE FIRST) ====================
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import json
import hashlib
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import math
import time
import shutil
import datetime
import zipfile
import logging

torch._logging.set_logs(inductor=logging.WARNING, dynamic=logging.WARNING)
logging.getLogger("torch._inductor").setLevel(logging.WARNING)

# ==================== 1. SETUP & OUTPUT CONFIG ====================
def ensure_drive_mounted():
    if os.path.isdir("/content/drive/MyDrive"):
        return
    try:
        from google.colab import drive
        print("Mounting Google Drive...")
        drive.mount("/content/drive", force_remount=False)
    except ImportError:
        print("Drive mount failed. Outputs will stay local.")

ensure_drive_mounted()

# ==================== MANIFEST HELPERS ====================
def file_hash(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while chunk := f.read(8192):
            h.update(chunk)
    return h.hexdigest()

def file_info(path):
    if not os.path.exists(path):
        return {"path": path, "exists": False}
    return {
        "path": path,
        "size_mb": round(os.path.getsize(path) / 1024 / 1024, 2),
        "sha256": file_hash(path),
    }

# ==================== 2. CONFIGURATION ====================
SPARKYLLM = "/content/drive/MyDrive/sparkyllm"
CHECKPOINT_DIR = os.path.join(SPARKYLLM, "checkpoints")
MANIFEST_DIR = os.path.join(SPARKYLLM, "manifests")
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(MANIFEST_DIR, exist_ok=True)

TOKEN_BIN = "/content/train_long.bin"
META_PATH = "/content/train_long_meta.json"

CHECKPOINT_NAME = "gpt_medium_phase2.pth"
CHECKPOINT_PATH = os.path.join(CHECKPOINT_DIR, CHECKPOINT_NAME)

# -- Archive old checkpoint if it exists --
if os.path.exists(CHECKPOINT_PATH):
    mod_time = os.path.getmtime(CHECKPOINT_PATH)
    date_str = datetime.datetime.fromtimestamp(mod_time).strftime("%Y%m%d_%H%M%S")
    archived_name = f"gpt_medium_phase2_{date_str}.pth"
    archived_path = os.path.join(CHECKPOINT_DIR, archived_name)
    if not os.path.exists(archived_path):
        shutil.copy2(CHECKPOINT_PATH, archived_path)
        print(f"Archived old checkpoint as: {archived_name}")

# -- MODEL ARCHITECTURE --
BLOCK_SIZE = 768
STRIDE = BLOCK_SIZE
EMBED_DIM = 1280
NUM_LAYERS = 24
NUM_HEADS = EMBED_DIM // 64
FF_HIDDEN_DIM = 4 * EMBED_DIM

# -- TRAINING HYPERPARAMETERS --
BATCH_SIZE = 32
GRAD_ACCUM_STEPS = 8
EPOCHS = 1
MAX_LR = 3e-4
MIN_LR = 3e-5
WEIGHT_DECAY = 0.1
WARMUP_STEPS = 100

# Hardware Settings
torch.set_float32_matmul_precision('high')
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == 'cuda':
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name} | VRAM: {props.total_memory / 1024**3:.1f} GB")

# ==================== 3. MODEL DEFINITION ====================

class CausalSelfAttention(nn.Module):
    def __init__(self, embed_dim, num_heads, dropout):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.c_attn = nn.Linear(embed_dim, 3 * embed_dim)
        self.c_proj = nn.Linear(embed_dim, embed_dim)
        self.dropout = dropout

    def forward(self, x):
        B, T, C = x.size()
        qkv = self.c_attn(x)
        q, k, v = qkv.split(C, dim=2)
        q = q.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True, dropout_p=self.dropout if self.training else 0.0)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.c_proj(y)

class SwiGLU(nn.Module):
    def __init__(self, embed_dim, hidden_dim):
        super().__init__()
        self.w1 = nn.Linear(embed_dim, hidden_dim * 2)
        self.w2 = nn.Linear(hidden_dim, embed_dim)
    def forward(self, x):
        x1, x2 = self.w1(x).chunk(2, dim=-1)
        return self.w2(F.silu(x1) * x2)

class TransformerBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, ff_hidden_dim, dropout):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn = CausalSelfAttention(embed_dim, num_heads, dropout)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.ff = SwiGLU(embed_dim, ff_hidden_dim)
        self.dropout = dropout
    def forward(self, x):
        x = x + F.dropout(self.attn(self.norm1(x)), p=self.dropout, training=self.training)
        x = x + F.dropout(self.ff(self.norm2(x)), p=self.dropout, training=self.training)
        return x

class SimpleGPT(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, EMBED_DIM)
        self.position_embedding = nn.Embedding(BLOCK_SIZE, EMBED_DIM)
        self.blocks = nn.ModuleList([TransformerBlock(EMBED_DIM, NUM_HEADS, FF_HIDDEN_DIM, 0.1) for _ in range(NUM_LAYERS)])
        self.final_norm = nn.LayerNorm(EMBED_DIM)
        self.lm_head = nn.Linear(EMBED_DIM, vocab_size, bias=False)
        self.token_embedding.weight = self.lm_head.weight
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None: torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx):
        B, T = idx.size()
        pos = torch.arange(T, device=idx.device)
        x = self.token_embedding(idx) + self.position_embedding(pos)
        for block in self.blocks: x = block(x)
        return self.lm_head(self.final_norm(x))

# ==================== 4. DATASET & INITIALIZATION ====================
class TokenDataset(Dataset):
    def __init__(self, bin_path, block_size, stride):
        self.tokens = np.memmap(bin_path, dtype=np.uint32, mode="r")
        self.block_size = block_size
        self.stride = stride
        self.length = (len(self.tokens) - block_size - 1) // stride
    def __len__(self): return self.length
    def __getitem__(self, idx):
        s = idx * self.stride
        if s + self.block_size + 1 > len(self.tokens): s = len(self.tokens) - self.block_size - 1
        chunk = self.tokens[s : s + self.block_size + 1].astype(np.int64)
        return torch.from_numpy(chunk[:-1]), torch.from_numpy(chunk[1:])

# Load Meta
with open(META_PATH, "r") as f:
    meta = json.load(f)
VOCAB_SIZE = int(meta["vocab_size"])

if VOCAB_SIZE % 64 != 0:
    VOCAB_SIZE = ((VOCAB_SIZE + 63) // 64) * 64
    print(f"Vocab padded to {VOCAB_SIZE} (Tensor Core Optimized)")

# -- Compute current tokenization_id from the tokenizer on disk --
tokenizer_on_disk = "/content/tokenizer.json"
if os.path.exists(tokenizer_on_disk):
    current_tok_id = file_hash(tokenizer_on_disk)[:16]
else:
    current_tok_id = None
    print("WARNING: tokenizer.json not found locally, cannot verify tokenization_id")

model = SimpleGPT(VOCAB_SIZE).to(device)

# --- CHECKPOINT LOADING (with tokenizer mismatch check) ---
resume_path = None
if os.path.exists(CHECKPOINT_PATH):
    # Check if checkpoint's tokenizer matches current tokenizer
    train_manifest_path = os.path.join(MANIFEST_DIR, "training_latest.json")
    safe_to_load = True
    if os.path.exists(train_manifest_path) and current_tok_id:
        with open(train_manifest_path) as f:
            train_manifest = json.load(f)
        ckpt_tok_id = train_manifest.get("tokenization_id")
        if ckpt_tok_id and ckpt_tok_id != current_tok_id:
            print(f"TOKENIZER MISMATCH — checkpoint was trained with tokenization_id={ckpt_tok_id}")
            print(f"  Current tokenizer has tokenization_id={current_tok_id}")
            print(f"  Skipping checkpoint, training from scratch.")
            safe_to_load = False
        else:
            print(f"Tokenizer match confirmed: {current_tok_id}")

    if safe_to_load:
        print(f"Resuming from: {CHECKPOINT_PATH}")
        resume_path = CHECKPOINT_PATH
        state_dict = torch.load(CHECKPOINT_PATH, map_location=device)
        new_state = {k.replace("_orig_mod.", ""): v for k, v in state_dict.items()}
        model_state = model.state_dict()
        safe_state = {}

        for k, v in new_state.items():
            if k in model_state and v.shape == model_state[k].shape:
                safe_state[k] = v

        model.load_state_dict(safe_state, strict=False)
        print(f"State loaded: {len(safe_state)} layers matched.")
else:
    print(f"No checkpoint found at {CHECKPOINT_PATH} — training from scratch.")

# Compile
print("Compiling model...")
model = torch.compile(model)

# Optimizer
param_dict = {pn: p for pn, p in model.named_parameters() if p.requires_grad}
decay_params = [p for n, p in param_dict.items() if p.dim() >= 2]
nodecay_params = [p for n, p in param_dict.items() if p.dim() < 2]
optim_groups = [{'params': decay_params, 'weight_decay': WEIGHT_DECAY},
                {'params': nodecay_params, 'weight_decay': 0.0}]
optimizer = optim.AdamW(optim_groups, lr=MAX_LR, fused=True)

# Loader
dataset = TokenDataset(TOKEN_BIN, BLOCK_SIZE, STRIDE)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True,
                    num_workers=4, pin_memory=True, prefetch_factor=2)
print(f"Data: {len(dataset):,} samples | Batch: {BATCH_SIZE} | Accum: {GRAD_ACCUM_STEPS}")
print(f"Effective Batch Size: {BATCH_SIZE * GRAD_ACCUM_STEPS}")

# ==================== 5. TRAINING LOOP ====================
def get_lr(it, total_it):
    if it < WARMUP_STEPS: return MAX_LR * it / WARMUP_STEPS
    decay_ratio = (it - WARMUP_STEPS) / max(1, (total_it - WARMUP_STEPS))
    coeff = 0.5 * (1.0 + math.cos(math.pi * min(1.0, decay_ratio)))
    return MIN_LR + coeff * (MAX_LR - MIN_LR)

curr_step = 0
total_steps = (len(loader) * EPOCHS) // GRAD_ACCUM_STEPS
final_loss = None

print(f"Total optimizer steps: {total_steps} | Warmup: {WARMUP_STEPS}")
print("\nSTARTING TRAINING (BFloat16)\n")

for epoch in range(1, EPOCHS + 1):
    model.train()
    pbar = tqdm(loader, desc=f"Ep {epoch}/{EPOCHS}", dynamic_ncols=True, mininterval=1.0)

    for batch_idx, (xb, yb) in enumerate(pbar):
        lr = get_lr(curr_step, total_steps)
        for pg in optimizer.param_groups: pg['lr'] = lr

        xb, yb = xb.to(device, non_blocking=True), yb.to(device, non_blocking=True)

        with torch.amp.autocast("cuda", dtype=torch.bfloat16):
            logits = model(xb)
            loss = F.cross_entropy(logits.view(-1, VOCAB_SIZE), yb.view(-1))
            loss = loss / GRAD_ACCUM_STEPS

        loss.backward()

        if (batch_idx + 1) % GRAD_ACCUM_STEPS == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)
            curr_step += 1
            final_loss = loss.item() * GRAD_ACCUM_STEPS
            pbar.set_postfix(loss=final_loss, lr=f"{lr:.2e}")

    # End of Epoch Save
    print(f"Saving Epoch {epoch}...")
    torch.save(model.state_dict(), CHECKPOINT_PATH)

# ==================== 6. FINALIZE ====================
print("Training Complete.")
torch.save(model.state_dict(), CHECKPOINT_PATH)

# ==================== 7. SAVE MANIFEST ====================
manifest = {
    "stage": "training",
    "created": datetime.datetime.now().isoformat(),
    "checkpoint": file_info(CHECKPOINT_PATH),
    "tokenization_id": current_tok_id,
    "config": {
        "block_size": BLOCK_SIZE, "stride": STRIDE,
        "embed_dim": EMBED_DIM, "num_layers": NUM_LAYERS,
        "num_heads": NUM_HEADS, "ff_hidden_dim": FF_HIDDEN_DIM,
        "dropout": 0.1, "vocab_size_padded": VOCAB_SIZE,
        "batch_size": BATCH_SIZE, "grad_accum_steps": GRAD_ACCUM_STEPS,
        "epochs": EPOCHS, "max_lr": MAX_LR, "min_lr": MIN_LR,
        "warmup_steps": WARMUP_STEPS, "weight_decay": WEIGHT_DECAY,
        "precision": "bfloat16", "optimizer": "AdamW (fused)",
        "grad_clip": 1.0, "lr_schedule": "cosine_with_warmup",
    },
    "results": {
        "final_loss": final_loss,
        "total_optimizer_steps": curr_step,
        "resumed_from": resume_path,
    },
}
manifest_path = os.path.join(MANIFEST_DIR, "training_latest.json")
with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2)
print(f"Manifest saved: {manifest_path}")
print(f"  tokenization_id: {current_tok_id}")
print(f"  final_loss: {final_loss}")

# ==================== 8. ZIP ====================
zip_name = f"gpt_medium_artifacts_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}.zip"
zip_path = os.path.join(CHECKPOINT_DIR, zip_name)

print(f"Zipping to {zip_path}...")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    if os.path.exists(CHECKPOINT_PATH): zf.write(CHECKPOINT_PATH, arcname=CHECKPOINT_NAME)
    if os.path.exists(META_PATH): zf.write(META_PATH, arcname="meta.json")

try:
    from google.colab import files
    print(f"Downloading zip...")
    files.download(zip_path)
except ImportError:
    print("Check Google Drive for the final zip file.")